# Tokenizer Deep Dive

End-to-end deep dive into tokenizers:

1. **What is tokenization?** — Why we need it, core concepts
2. **Byte Pair Encoding (BPE)** — Algorithm, step-by-step walkthrough
3. **BPE from Scratch** — Merges, building the vocab
4. **Tokenizer Training** — Picking the vocab size, the training process
5. **Real Tokenizer Comparison** — tiktoken (GPT-2) vs Qwen3.5 tokenizer
6. **Special Tokens** — Special tokens, chat templates, system / user / assistant
7. **Extending a Tokenizer** — Adding new tokens, updating embeddings
8. **Practical Topics** — Token count optimization, multilingual tokenization, edge cases

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

---
## 1. What Is Tokenization?

LLMs can't process text directly — they work with numbers.
Tokenization: the **text → token IDs** transform.

```
"Hello world" → [15339, 1917] → Embedding → Model → Logits → [next_token_id] → "!"
```

**Three approaches:**
- **Character-level**: each character is a token. Vocab is small (~256) but sequences get long.
- **Word-level**: each word is a token. Vocab gets huge, unknown words are problematic.
- **Subword (BPE)**: the middle path — frequent words become a single token, rare words get split.

In [ ]:
text = "Tokenization is fundamental to LLMs."

# Character-level
char_tokens = list(text)
print(f"Character-level : {len(char_tokens)} tokens")
print(f"  {char_tokens[:15]}...")

# Word-level
word_tokens = text.split()
print(f"\nWord-level      : {len(word_tokens)} tokens")
print(f"  {word_tokens}")

# Byte-level (UTF-8)
byte_tokens = list(text.encode("utf-8"))
print(f"\nByte-level      : {len(byte_tokens)} tokens")
print(f"  {byte_tokens[:15]}...")
print(f"  Each byte 0-255 → vocab size = 256")

In [ ]:
# UTF-8 encoding: byte count varies across languages
examples = [
    ("Hello", "English (ASCII)"),
    ("Merhaba", "Turkish"),
    ("你好", "Chinese"),
    ("🤖", "Emoji"),
]

for text, lang in examples:
    bytes_repr = list(text.encode("utf-8"))
    print(f"  '{text}' ({lang}): {len(text)} chars, {len(bytes_repr)} bytes → {bytes_repr}")

---
## 2. Byte Pair Encoding (BPE) — The Algorithm

BPE iteratively merges the most frequent pair of bytes:

1. Convert the text into bytes (256 base tokens)
2. Find the most frequent adjacent pair
3. Merge that pair into a new token (add it to the vocab)
4. Repeat 2-3 until you reach the desired vocab size

This produces a list of **merge rules**. During encoding, the rules are applied in order.

In [ ]:
# Step-by-step BPE demo
text = "aaabdaaabac"
tokens = list(text.encode("utf-8"))  # to bytes

print(f"Start: {tokens}")
print(f"       {''.join(chr(t) for t in tokens)}")
print()

def get_most_frequent_pair(token_list):
    pairs = Counter(zip(token_list[:-1], token_list[1:]))
    return pairs.most_common(1)[0] if pairs else None

def merge_pair(token_list, pair, new_id):
    result = []
    i = 0
    while i < len(token_list):
        if i < len(token_list) - 1 and (token_list[i], token_list[i+1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(token_list[i])
            i += 1
    return result

vocab = {i: bytes([i]) for i in range(256)}
merges = []
next_id = 256

for step in range(5):
    result = get_most_frequent_pair(tokens)
    if result is None:
        break
    pair, count = result

    vocab[next_id] = vocab[pair[0]] + vocab[pair[1]]
    merges.append(pair)
    tokens = merge_pair(tokens, pair, next_id)

    pair_repr = vocab[pair[0]].decode() + vocab[pair[1]].decode()
    print(f"Step {step+1}: merge ({pair[0]},{pair[1]}) = '{pair_repr}' → id={next_id}, count={count}")
    print(f"  Tokens: {tokens}")
    print(f"  Length: {len(tokens)}")
    print()

    next_id += 1

print(f"Merge rules: {merges}")
print(f"Vocab size: 256 (base) + {len(merges)} (merges) = {256 + len(merges)}")

---
## 3. BPE from Scratch

A simple, working BPE tokenizer. Two stages:
- **Training**: learn merge rules from text
- **Encoding/Decoding**: apply the merge rules to tokenize/detokenize

In [ ]:
class SimpleBPE:
    def __init__(self):
        self.merges = {}       # (p1, p2) -> new_id
        self.vocab = {}        # id -> bytes

    def train(self, text, vocab_size):
        """BPE training: learn the merge rules."""
        assert vocab_size >= 256
        tokens = list(text.encode("utf-8"))

        # Base vocab: 0-255 (single bytes)
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = {}

        num_merges = vocab_size - 256
        for i in range(num_merges):
            # Most frequent pair
            pairs = Counter(zip(tokens[:-1], tokens[1:]))
            if not pairs:
                break
            top_pair = pairs.most_common(1)[0][0]

            new_id = 256 + i
            self.merges[top_pair] = new_id
            self.vocab[new_id] = self.vocab[top_pair[0]] + self.vocab[top_pair[1]]
            tokens = merge_pair(tokens, top_pair, new_id)

        return len(self.merges)

    def encode(self, text):
        """Convert text into a list of token IDs."""
        tokens = list(text.encode("utf-8"))
        # Apply merge rules in order
        for pair, new_id in self.merges.items():
            tokens = merge_pair(tokens, pair, new_id)
        return tokens

    def decode(self, token_ids):
        """Convert a list of token IDs back into text."""
        byte_seq = b"".join(self.vocab[tid] for tid in token_ids)
        return byte_seq.decode("utf-8", errors="replace")

In [ ]:
# Download training data
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
if not os.path.exists(file_path):
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(resp.text)

with open(file_path, "r", encoding="utf-8") as f:
    train_text = f.read()

print(f"Training text: {len(train_text):,} characters")

# BPE training
bpe = SimpleBPE()
num_merges = bpe.train(train_text, vocab_size=500)
print(f"Number of merges: {num_merges}")
print(f"Vocab size: {len(bpe.vocab)}")

In [ ]:
# Encode / decode test
test_texts = [
    "Hello world",
    "The quick brown fox jumps over the lazy dog.",
    "Tokenization is important!",
]

for text in test_texts:
    ids = bpe.encode(text)
    decoded = bpe.decode(ids)
    tokens = [bpe.decode([tid]) for tid in ids]

    print(f"'{text}'")
    print(f"  IDs    : {ids}")
    print(f"  Tokens : {tokens}")
    print(f"  Decoded: '{decoded}'")
    print(f"  Compression: {len(text.encode('utf-8'))} bytes → {len(ids)} tokens ({len(text.encode('utf-8')) / len(ids):.1f}x)")
    print()

---
## 4. Tokenizer Training

Choosing the vocab size is critical:
- **Small vocab** (256-1K): too many tokens, long sequences, slow training
- **Large vocab** (100K+): short sequences but a huge embedding table
- **Typical values**: GPT-2: 50,257 | Llama: 32,000 | Qwen3.5: 248,320

In [ ]:
# Effect of vocab size on token count
test_text = "The quick brown fox jumps over the lazy dog. " * 10
vocab_sizes = [260, 300, 400, 500, 750, 1000]

results = []
for vs in vocab_sizes:
    t = SimpleBPE()
    t.train(train_text, vocab_size=vs)
    ids = t.encode(test_text)
    results.append((vs, len(ids)))
    print(f"  vocab_size={vs:>5d} → {len(ids):>4d} tokens (compression: {len(test_text.encode('utf-8')) / len(ids):.1f}x)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r[0] for r in results], [r[1] for r in results], marker="o", color="steelblue")
ax.set_xlabel("Vocab Size")
ax.set_ylabel("Token Count")
ax.set_title("Vocab Size vs Token Count (same text)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Real Tokenizer Comparison: tiktoken vs Qwen3.5

- **tiktoken** (GPT-2): OpenAI BPE, 50,257 vocab
- **Qwen3.5 tokenizer**: HuggingFace BPE, 248,320 vocab

In [ ]:
import tiktoken
from transformers import AutoTokenizer

# GPT-2 tokenizer (tiktoken)
gpt2_tok = tiktoken.get_encoding("gpt2")

# Qwen3.5 tokenizer
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B-Base", trust_remote_code=True)

print(f"GPT-2 vocab size : {gpt2_tok.n_vocab:,}")
print(f"Qwen3.5 vocab size: {qwen_tok.vocab_size:,}")

In [ ]:
# Tokenize the same texts with both tokenizers
test_texts = [
    ("Hello, how are you today?", "English"),
    ("Yapay zeka dünyayı değiştiriyor.", "Turkish"),
    ("人工智能正在改变世界", "Chinese"),
    ("def fibonacci(n):\n    if n < 2:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)", "Code"),
    ("https://www.example.com/api/v2/users?page=1&limit=10", "URL"),
    ("2024-03-15T10:30:00Z", "Date"),
]

print(f"{'Text':<30s} {'Lang':<10s} {'GPT-2':>6s} {'Qwen':>6s} {'Diff':>6s}")
print("-" * 65)

for text, lang in test_texts:
    gpt2_ids = gpt2_tok.encode(text)
    qwen_ids = qwen_tok.encode(text)
    diff = len(gpt2_ids) - len(qwen_ids)
    sign = "+" if diff > 0 else ""
    display = text[:28] + ".." if len(text) > 30 else text
    print(f"{display:<30s} {lang:<10s} {len(gpt2_ids):>6d} {len(qwen_ids):>6d} {sign}{diff:>5d}")

In [ ]:
# Detailed token comparison
text = "Yapay zeka dünyayı değiştiriyor."

print(f"=== GPT-2 tokenization ===")
gpt2_ids = gpt2_tok.encode(text)
for tid in gpt2_ids:
    print(f"  {tid:>6d} → '{gpt2_tok.decode([tid])}'")

print(f"\n=== Qwen3.5 tokenization ===")
qwen_ids = qwen_tok.encode(text)
for tid in qwen_ids:
    print(f"  {tid:>6d} → '{qwen_tok.decode([tid])}'")

print(f"\nGPT-2 : {len(gpt2_ids)} tokens")
print(f"Qwen  : {len(qwen_ids)} tokens")

In [ ]:
# Token fertility: tokens per character (lower = more efficient)
languages = {
    "English": "Artificial intelligence is transforming the world in unprecedented ways.",
    "Turkish": "Yapay zeka dünyayı benzeri görülmemiş şekillerde dönüştürüyor.",
    "Chinese": "人工智能正在以前所未有的方式改变世界。",
    "Arabic": "الذكاء الاصطناعي يغير العالم بطرق غير مسبوقة.",
    "Code": "for i in range(10): print(f'Hello {i}')",
}

print(f"{'Language':<10s} {'GPT-2 fertility':>18s} {'Qwen fertility':>18s}")
print("-" * 50)
for lang, text in languages.items():
    g_ids = gpt2_tok.encode(text)
    q_ids = qwen_tok.encode(text)
    g_fertility = len(g_ids) / len(text)
    q_fertility = len(q_ids) / len(text)
    print(f"{lang:<10s} {g_fertility:>14.3f} tok/chr {q_fertility:>14.3f} tok/chr")

---
## 6. Special Tokens

Special tokens skip the BPE merge process — they are recognized as a single token directly.

- `<|endoftext|>` — end of sequence
- `<|im_start|>`, `<|im_end|>` — chat template
- `<think>`, `</think>` — reasoning mode (Qwen3)
- `<|vision_start|>` — multimodal tokens

In [ ]:
print("=== Qwen3.5 Special Tokens ===")
print(f"All special tokens: {qwen_tok.all_special_tokens}")
print(f"\nKey tokens:")
for name in ['bos_token', 'eos_token', 'pad_token', 'unk_token']:
    token = getattr(qwen_tok, name)
    token_id = getattr(qwen_tok, f"{name}_id")
    print(f"  {name:<12s}: '{token}' (id={token_id})")

# Encoding special tokens
print(f"\n=== Special Token Encoding ===")
special_texts = [
    "Normal text",
    "<|im_start|>user\nHello<|im_end|>",
]
for text in special_texts:
    ids = qwen_tok.encode(text)
    tokens = [qwen_tok.decode([tid]) for tid in ids]
    print(f"  '{text[:50]}'")
    print(f"    → {ids}")
    print(f"    → {tokens}")

In [ ]:
# Chat template: the actual input format fed to the model
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is BPE?"},
]

if hasattr(qwen_tok, 'apply_chat_template'):
    chat_text = qwen_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    print("=== Chat Template Output ===")
    print(repr(chat_text))

    print(f"\n=== Tokenized ===")
    chat_ids = qwen_tok.encode(chat_text)
    print(f"Token count: {len(chat_ids)}")
    print(f"First 20 tokens:")
    for tid in chat_ids[:20]:
        print(f"  {tid:>6d} → '{qwen_tok.decode([tid])}'")
else:
    print("No chat template available.")

---
## 7. Extending a Tokenizer

When you want to add a new special token:
1. Add the new token to the tokenizer
2. Resize the model's embedding layer
3. (If weight tying is on) lm_head is updated alongside

This matters in particular when adding domain-specific tokens (e.g. `<|code_start|>`, `<|tool_call|>`).

In [ ]:
# Adding a new special token to tiktoken
base_tok = tiktoken.get_encoding("gpt2")

# New tokens
custom_tokens = {"<|tool_call|>": base_tok.n_vocab, "<|tool_result|>": base_tok.n_vocab + 1}

extended_tok = tiktoken.Encoding(
    name="gpt2_extended",
    pat_str=base_tok._pat_str,
    mergeable_ranks=base_tok._mergeable_ranks,
    special_tokens={**base_tok._special_tokens, **custom_tokens},
)

print(f"Original vocab: {base_tok.n_vocab:,}")
print(f"Extended vocab: {extended_tok.n_vocab:,}")

# Test
text = "Call function <|tool_call|> get_weather <|tool_result|> sunny"
ids = extended_tok.encode(text, allowed_special=set(custom_tokens.keys()))
print(f"\nEncoded: {ids}")
for tid in ids:
    print(f"  {tid:>6d} → '{extended_tok.decode([tid])}'")

In [ ]:
# Resize a model's embeddings (Ch05 approach)
emb_dim = 768
old_vocab_size = 50257
new_vocab_size = 50259  # +2 new tokens

# Simulate the old embedding
old_embedding = torch.nn.Embedding(old_vocab_size, emb_dim)

# New embedding: enlarged
new_embedding = torch.nn.Embedding(new_vocab_size, emb_dim)
new_embedding.weight.data[:old_vocab_size] = old_embedding.weight.data
# New tokens start with random init

print(f"Old embedding: {old_embedding}")
print(f"New embedding: {new_embedding}")
print(f"\nNew token embeddings (random init):")
print(f"  Token {old_vocab_size}: {new_embedding.weight.data[old_vocab_size, :5].tolist()}")
print(f"  Token {old_vocab_size+1}: {new_embedding.weight.data[old_vocab_size+1, :5].tolist()}")
print(f"\nThese new tokens learn meaningful embeddings via fine-tuning.")

---
## 8. Practical Topics

### 8.1 Edge Cases

In [ ]:
# Edge cases: tokenizer corner cases
edge_cases = [
    "",                          # Empty string
    " ",                         # Single space
    "\n\n\n",                    # Newlines
    "aaaaaaaaaaaa",              # Repeated character
    "a" * 100,                   # Very long repetition
    "🤖🤖🤖",                  # Repeated emoji
    "123456789",                 # Numbers
    "    def foo():    ",        # Code whitespace
    "<html><body></body></html>",# HTML
]

print(f"{'Text':<35s} {'GPT-2':>6s} {'Qwen':>6s}")
print("-" * 50)
for text in edge_cases:
    g = gpt2_tok.encode(text)
    q = qwen_tok.encode(text)
    display = repr(text)[:33]
    print(f"{display:<35s} {len(g):>6d} {len(q):>6d}")

### 8.2 Token Count and Cost

When you use an API, token count = cost. A more efficient tokenizer = cheaper.

In [ ]:
# Token count comparison on a larger document
with open(file_path, "r", encoding="utf-8") as f:
    long_text = f.read()

g_ids = gpt2_tok.encode(long_text)
q_ids = qwen_tok.encode(long_text)

print(f"Text length     : {len(long_text):,} characters")
print(f"GPT-2 tokens    : {len(g_ids):,}")
print(f"Qwen3.5 tokens  : {len(q_ids):,}")
print(f"Qwen savings    : {(1 - len(q_ids)/len(g_ids))*100:.1f}%")
print(f"\nAverage token length:")
print(f"  GPT-2 : {len(long_text) / len(g_ids):.1f} chars/token")
print(f"  Qwen  : {len(long_text) / len(q_ids):.1f} chars/token")

### 8.3 Tokenization and Model Performance

Tokenizer choice directly impacts model performance:
- **Fertility ratio**: how many tokens does the same text produce? Lower = better.
- **Multilingual support**: does the vocab contain Turkish/Chinese tokens? If not, every word is fragmented.
- **Code support**: how are spaces, indentation, and special symbols tokenized?

In [ ]:
# Per-language fertility visualization
languages = {
    "English": "The quick brown fox jumps over the lazy dog.",
    "Turkish": "Hızlı kahverengi tilki tembel köpeğin üzerinden atlar.",
    "Chinese": "快速的棕色狐狸跳过了懒惰的狗。",
    "Arabic": "الثعلب البني السريع يقفز فوق الكلب الكسول.",
    "Russian": "Быстрая коричневая лиса перепрыгивает через ленивую собаку.",
    "Python": "def quick_brown_fox(): return 'jumps over lazy dog'",
    "JSON": '{"fox": "brown", "action": "jump", "over": "dog"}',
}

gpt2_ferts = []
qwen_ferts = []
langs = []

for lang, text in languages.items():
    g = len(gpt2_tok.encode(text)) / len(text)
    q = len(qwen_tok.encode(text)) / len(text)
    gpt2_ferts.append(g)
    qwen_ferts.append(q)
    langs.append(lang)

x = np.arange(len(langs))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, gpt2_ferts, width, label="GPT-2", color="steelblue")
ax.bar(x + width/2, qwen_ferts, width, label="Qwen3.5", color="coral")
ax.set_ylabel("Fertility (tokens/char, lower = better)")
ax.set_xticks(x)
ax.set_xticklabels(langs, rotation=30)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title("Tokenizer Fertility Comparison")
plt.tight_layout()
plt.show()

---
## Summary

| Topic | Detail |
|-------|--------|
| **BPE Algorithm** | Iteratively merge byte pairs → merge rules |
| **Training** | Learn merge rules over a large corpus |
| **Encoding** | Apply merge rules in order |
| **Decoding** | Token IDs → byte sequence → text |
| **Vocab Size** | Small = long sequences, Large = huge embedding table |
| **Special Tokens** | Skip BPE, recognized directly |
| **Extending** | New token + embedding resize + fine-tune |
| **Fertility** | Tokens-per-character ratio, depends on language and tokenizer |

**Key takeaways:**
- The tokenizer is the most important component the model never "sees" — bad tokenizer = bad model
- Vocab size is a trade-off: shorter sequences vs larger embedding table
- Multilingual models need a large vocab (Qwen: 248K vs GPT-2: 50K)
- When adding tokens, the embedding and lm_head must be updated together